In [ ]:
import pandas as pd#read csv,parquet
import numpy as np#for scientific computation of matrices
from xgboost import XGBRegressor,XGBClassifier


import warnings#avoid some negligible errors
#The filterwarnings () method is used to set warning filters, which can control the output method and level of warning information.
warnings.filterwarnings('ignore')
import random#provide some function to generate random_seed.
#set random seed,to make sure model can be recurrented.
def seed_everything(seed):
    np.random.seed(seed)#numpy's random seed
    random.seed(seed)#python built-in random seed
seed_everything(seed=2026)

In [2]:
target_col='Irrigation_Need'
num_classes=3

drop_cols=['id']

train=pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/train.csv").drop(drop_cols,axis=1)
test=pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/test.csv").drop(drop_cols,axis=1)
print(f"train.shape:{train.shape},test.shape:{test.shape}")
target2idx={v:i for i,v in enumerate(train[target_col].unique())}
idx2target={v:i for i,v in target2idx.items()}
for df in [train]:
    df[target_col]=df[target_col].map(target2idx)


CATS=[c for c in test.columns if train[c].dtype==object]
NUMS=[c for c in test.columns if c not in CATS]

train.head()

train.shape:(630000, 20),test.shape:(270000, 19)


,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,0
1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,0
2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,0
3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,1
4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,0


In [3]:
M=train[NUMS].max()
def FE(df): 
    
    for c in NUMS:
        for k in range(-4,4):
            df[f"{c}_digit{k}"] = (df[c] // (10**k) % 10).astype('int8')

        if M[c]<10:
            df[c]=df[c].round(3)
        elif M[c]<100:
            df[c]=df[c].round(2)
        else:
            df[c]=df[c].round(1)
    return df 

for df in [train,test]:
    df=FE(df)

In [4]:
DROP=[c for c in test.columns if test[c].nunique()==1]
print(f"DROP:{DROP}")
train.drop(DROP,axis=1,inplace=True)
test.drop(DROP,axis=1,inplace=True)

CATEGORY=CATS+[c for c in test.columns if 'digit' in c]
for c in CATEGORY:

    freq = train[c].value_counts()
    mapping = {val: idx for idx, (val, count) in enumerate(freq[freq >= 5].items())}
    mapping_default = len(mapping)
    train[c] = train[c].map(lambda x: mapping.get(x, mapping_default))
    test[c] = test[c].map(lambda x: mapping.get(x, mapping_default))
    

FEATURES=CATEGORY+NUMS
train.head()

DROP:['Soil_pH_digit1', 'Soil_pH_digit2', 'Soil_pH_digit3', 'Soil_Moisture_digit2', 'Soil_Moisture_digit3', 'Organic_Carbon_digit1', 'Organic_Carbon_digit2', 'Organic_Carbon_digit3', 'Electrical_Conductivity_digit1', 'Electrical_Conductivity_digit2', 'Electrical_Conductivity_digit3', 'Temperature_C_digit2', 'Temperature_C_digit3', 'Humidity_digit2', 'Humidity_digit3', 'Sunlight_Hours_digit2', 'Sunlight_Hours_digit3', 'Wind_Speed_kmh_digit2', 'Wind_Speed_kmh_digit3', 'Field_Area_hectare_digit2', 'Field_Area_hectare_digit3', 'Previous_Irrigation_mm_digit3']


,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,...,Field_Area_hectare_digit-1,Field_Area_hectare_digit0,Field_Area_hectare_digit1,Previous_Irrigation_mm_digit-4,Previous_Irrigation_mm_digit-3,Previous_Irrigation_mm_digit-2,Previous_Irrigation_mm_digit-1,Previous_Irrigation_mm_digit0,Previous_Irrigation_mm_digit1,Previous_Irrigation_mm_digit2
0,2,4.92,32.58,1.01,3.05,15.01,50.61,726.0,5.90,16.79,...,3,4,0,0,0,0,6,3,0,1
1,1,7.08,56.61,0.44,2.00,22.92,67.86,985.7,6.98,3.39,...,6,6,0,0,0,0,6,5,5,0
2,1,5.69,27.71,0.81,2.83,26.97,92.22,2201.7,6.05,3.85,...,6,8,0,0,0,6,8,2,0,1
3,0,5.65,13.32,1.33,0.87,13.32,61.57,1357.3,9.12,2.31,...,4,8,0,0,1,0,9,4,3,0
4,1,7.96,59.14,0.38,0.96,20.22,91.11,1538.2,6.95,13.94,...,4,7,0,0,0,9,6,4,4,0


In [5]:
unique, counts = np.unique(train[target_col].values, return_counts=True)
count_dict = dict(zip(unique, counts))
avg_count = len(train) / len(unique)
weights_dict = {cls: avg_count / cnt for cls, cnt in count_dict.items()}
sample_weights = np.array([weights_dict[y] for y in train[target_col]])

In [6]:
def accuracy_score(t,p):
    if len(p.shape)==2:
        p=np.argmax(p,axis=1)
    C=3
    acc=0.0
    for i in range(C):
        acc+=np.sum((t==i)&(p==i))/np.sum(t==i)/C
    return acc      

In [7]:
class OrderedTE():
    def __init__(self, a=1):
        self.a = a
        
    def fit(self, train, category_cols=[], target_col='target'):
        self.train = train
        self.target_col = target_col
        self.category_cols = category_cols
        
        # 获取所有类别
        self.classes_ = sorted(train[target_col].unique())
        self.num_classes_ = len(self.classes_)
        
        # 全局先验（每个类别的概率）
        self.global_prior_ = train[target_col].value_counts(normalize=True).sort_index().values
        
        for c in self.category_cols:
            # 存储每个类别的统计信息
            stats_list = []
            
            # 为每个类别创建一列 TE 特征
            for k, cls in enumerate(self.classes_):
                # 创建指示变量：当前样本是否属于类别 cls
                y_binary = (train[target_col] == cls).astype(int)
                
                # 计算累积统计（ordered）
                df = train[[c]].copy()
                df['y'] = y_binary.values
                df['cnt'] = 1
                
                # 按类别分组计算累积和（不包括当前行）
                df['cum_cnt'] = df.groupby(c)['cnt'].cumsum() - df['cnt']
                df['cum_sum'] = df.groupby(c)['y'].cumsum() - df['y']
                
                # 平滑先验
                smooth_prior = self.a * self.global_prior_[k]
                
                # 计算 TE
                te_col = f'{c}_TE_cls{cls}'
                df[te_col] = (df['cum_sum'] + smooth_prior) / (df['cum_cnt'] + self.a)
                # 第一次出现的处理
                df.loc[df['cum_cnt'] == -1, te_col] = self.global_prior_[k]
                
                self.train[te_col] = df[te_col].values
                
                # 修正：使用 df（包含'y'列）来分组统计，而不是原始的 train
                stats_df = df.groupby(c)['y'].agg(['count', 'sum']).reset_index()
                stats_df.columns = [c, f'{c}_count_cls{cls}', f'{c}_sum_cls{cls}']
                stats_df[f'{c}_prior_cls{cls}'] = self.global_prior_[k]
                stats_list.append(stats_df)
            
            # 合并所有类别的统计
            combined_stats = stats_list[0]
            for i in range(1, len(stats_list)):
                combined_stats = combined_stats.merge(stats_list[i], on=c, how='outer')
            setattr(self, f'{c}_stats', combined_stats)
            
        return self.train
    
    def transform(self, test):
        for c in self.category_cols:
            stats_df = getattr(self, f'{c}_stats')
            test = test.merge(stats_df, on=c, how='left')
            
            for k, cls in enumerate(self.classes_):
                te_col = f'{c}_TE_cls{cls}'
                count_col = f'{c}_count_cls{cls}'
                sum_col = f'{c}_sum_cls{cls}'
                prior_col = f'{c}_prior_cls{cls}'
                
                if count_col in test.columns:
                    test[te_col] = (test[sum_col] + self.a * test[prior_col]) / (test[count_col] + self.a)
                    test[te_col] = test[te_col].fillna(test[prior_col])
                    test.drop([count_col, sum_col, prior_col], axis=1, inplace=True)
                else:
                    # 未见过的类别，使用全局先验
                    test[te_col] = self.global_prior_[k]
            
        return test

In [8]:
def reduce_mem_usage(df:pd.DataFrame, float16_as32:bool=True)->pd.DataFrame:
    #memory_usage()是df每列的内存使用量,sum是对它们求和, B->KB->MB
    start_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object and str(col_type)!='category':#num_col
            c_min,c_max = df[col].min(),df[col].max()
            if str(col_type)[:3] == 'int':#如果是int类型的变量,不管是int8,int16,int32还是int64
                #如果这列的取值范围是在int8的取值范围内,那就对类型进行转换 (-128 到 127)
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                #如果这列的取值范围是在int16的取值范围内,那就对类型进行转换(-32,768 到 32,767)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                #如果这列的取值范围是在int32的取值范围内,那就对类型进行转换(-2,147,483,648到2,147,483,647)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                #如果这列的取值范围是在int64的取值范围内,那就对类型进行转换(-9,223,372,036,854,775,808到9,223,372,036,854,775,807)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:#如果是浮点数类型.
                #如果数值在float16的取值范围内,如果觉得需要更高精度可以考虑float32
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    if float16_as32:#如果数据需要更高的精度可以选择float32
                        df[col] = df[col].astype(np.float32)
                    else:
                        df[col] = df[col].astype(np.float16)  
                #如果数值在float32的取值范围内，对它进行类型转换
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                #如果数值在float64的取值范围内，对它进行类型转换
                else:
                    df[col] = df[col].astype(np.float64)
    #calculate memory after optimization
    end_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage after optimization is: {:.2f} MB'.format(end_mem))
    print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))
    return df

In [9]:
from sklearn.model_selection import KFold

def xgb_eval_metric(y_true,y_pred):
    score=accuracy_score(y_true,y_pred)
    return score

X=train.drop([target_col],axis=1)
y=train[target_col]
test_X=test.copy()
oof_preds=np.zeros((len(y),num_classes))
test_preds=np.zeros((len(test_X),num_classes))

xgb_params = {
    'max_depth': 4,
    'colsample_bytree': 0.8,
    'subsample': 0.8,
    'n_estimators': 512,
    'learning_rate': 0.1,
    'early_stopping_rounds': 1024,
    'random_state': 2026,
    'n_jobs': -1,
    'enable_categorical': True,
    'alpha': 5,
    'reg_lambda': 5,
    'max_leaves': 30,
    'min_child_weight': 2,
    'tree_method': 'hist',
    'max_bin': 10000,
   # 'eval_metric':xgb_eval_metric,
    'device':'cuda',
}


n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

auc_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"\nFold {fold+1}/{n_folds}")
    
    # 划分数据
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    train_weights=sample_weights[train_idx]

    te=OrderedTE()
    full_df=pd.concat((X_train,y_train),axis=1)
    full_df['weight']=train_weights
    te_train=pd.concat([te.fit(full_df.sample(frac=1,random_state=42+i),
                               category_cols=FEATURES,target_col=target_col) for i in range(4)])
    X_train=te_train.drop([target_col,'weight'],axis=1)
    y_train=te_train[target_col]
    train_weights=te_train['weight']
    
    
    X_val=te.transform(X_val)
    X_test=te.transform(test_X)

    X_train.drop(CATS,axis=1,inplace=True)
    X_val.drop(CATS,axis=1,inplace=True)
    X_test.drop(CATS,axis=1,inplace=True)

    X_train=reduce_mem_usage(X_train)
    X_val=reduce_mem_usage(X_val)
    X_test=reduce_mem_usage(X_test)

    model=XGBClassifier(**xgb_params)
    
    model.fit(X_train,y_train,eval_set=[(X_val, y_val)],
                                  sample_weight=train_weights,
                                  verbose=100)
    
    y_pred = model.predict_proba(X_val)
    
    oof_preds[val_idx]=y_pred
    test_preds += model.predict_proba(X_test) / n_folds

print(f"oof CV:{accuracy_score(y,oof_preds)}")
np.save('oof_preds.npy',oof_preds)
np.save('test_preds.npy',test_preds)


Fold 1/5
Memory usage of dataframe is 5121.83 MB
Memory usage after optimization is: 2187.93 MB
Decreased by 57.3%
Memory usage of dataframe is 319.15 MB
Memory usage after optimization is: 135.78 MB
Decreased by 57.5%
Memory usage of dataframe is 683.90 MB
Memory usage after optimization is: 290.97 MB
Decreased by 57.5%
[0]	validation_0-mlogloss:0.99786
[100]	validation_0-mlogloss:0.07780
[200]	validation_0-mlogloss:0.05999
[300]	validation_0-mlogloss:0.05499
[400]	validation_0-mlogloss:0.05240
[500]	validation_0-mlogloss:0.05074
[511]	validation_0-mlogloss:0.05058

Fold 2/5
Memory usage of dataframe is 5121.83 MB
Memory usage after optimization is: 2187.93 MB
Decreased by 57.3%
Memory usage of dataframe is 319.15 MB
Memory usage after optimization is: 135.78 MB
Decreased by 57.5%
Memory usage of dataframe is 683.90 MB
Memory usage after optimization is: 290.97 MB
Decreased by 57.5%
[0]	validation_0-mlogloss:0.99699
[100]	validation_0-mlogloss:0.07679
[200]	validation_0-mlogloss:0.05

In [10]:
import optuna
from optuna.samplers import TPESampler

def objective(trial):
    # 类别权重（3个类别）
    cw1 = trial.suggest_float('cw1', 0.5, 3.0)  # 类别0的权重
    cw2 = trial.suggest_float('cw2', 0.5, 3.0)  # 类别1的权重
    cw3 = trial.suggest_float('cw3', 0.5, 3.0)  # 类别2的权重
    
    # 应用类别权重
    class_weights = np.array([cw1, cw2, cw3])
    adjusted_probs = oof_preds * class_weights
    
    # 重新归一化
    adjusted_probs = adjusted_probs / adjusted_probs.sum(axis=1, keepdims=True)
    
    # 计算准确率
    acc = accuracy_score(y, np.argmax(adjusted_probs, axis=1))
    
    return acc

# 创建Optuna研究
study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    study_name='class_weight_optimization'
)

[I 2026-04-08 10:37:36,230] A new study created in memory with name: class_weight_optimization


In [11]:
# 运行优化
study.optimize(objective, n_trials=200, show_progress_bar=True)

print(f"最佳准确率: {study.best_value:.6f}")
print("\n类别权重:")
print(f"  class_0 = {study.best_params['cw1']:.4f}")
print(f"  class_1 = {study.best_params['cw2']:.4f}")
print(f"  class_2 = {study.best_params['cw3']:.4f}")

# 使用最佳参数生成测试集预测
best_cw = np.array([study.best_params['cw1'], 
                    study.best_params['cw2'], 
                    study.best_params['cw3']])

# 应用类别权重
final_test_probs = test_preds * best_cw
final_test_probs = final_test_probs / final_test_probs.sum(axis=1, keepdims=True)

# 最终预测
final_test_preds = np.argmax(final_test_probs, axis=1)

  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-04-08 10:37:36,311] Trial 0 finished with value: 0.9781520835574806 and parameters: {'cw1': 1.4363502971184063, 'cw2': 2.87678576602479, 'cw3': 2.3299848545285125}. Best is trial 0 with value: 0.9781520835574806.
[I 2026-04-08 10:37:36,359] Trial 1 finished with value: 0.9789942584643676 and parameters: {'cw1': 1.9966462104925915, 'cw2': 0.8900466011060912, 'cw3': 0.8899863008405067}. Best is trial 1 with value: 0.9789942584643676.
[I 2026-04-08 10:37:36,412] Trial 2 finished with value: 0.9754977964709772 and parameters: {'cw1': 0.6452090304204987, 'cw2': 2.665440364437338, 'cw3': 2.002787529358022}. Best is trial 1 with value: 0.9789942584643676.
[I 2026-04-08 10:37:36,461] Trial 3 finished with value: 0.9769549713692881 and parameters: {'cw1': 2.2701814444901136, 'cw2': 0.5514612357395061, 'cw3': 2.9247746304049858}. Best is trial 1 with value: 0.9789942584643676.
[I 2026-04-08 10:37:36,513] Trial 4 finished with value: 0.9788321237743081 and parameters: {'cw1': 2.5811066020

In [12]:
sub=pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv")
sub[target_col]=final_test_preds
sub[target_col]=sub[target_col].map(idx2target)
sub.to_csv("xgb_with_orderedTE.csv",index=None)
sub.head()

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low
